In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import ta

In [15]:
def calculate_rsi(series: pd.Series, window: int = 14) -> pd.Series:
    """Calcula o Relative Strength Index (RSI)"""
    delta = series.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

In [16]:
data_dir = Path("../data/02_intermediate")

In [17]:
btc = pd.read_parquet(data_dir / "historical_data_btc.parquet")
fed = pd.read_parquet(data_dir / "us_fed_rate.parquet")
sp500 = pd.read_parquet(data_dir / "SP500.parquet")
fng = pd.read_parquet(data_dir / "greed_and_fear.parquet")
djia = pd.read_parquet(data_dir / "DJIA.parquet")
djca = pd.read_parquet(data_dir / "DJCA.parquet")

In [18]:
btc['date'] = pd.to_datetime(btc['formatted_date'])
fed['date'] = pd.to_datetime(fed['Release Date'])
sp500['date'] = pd.to_datetime(sp500['observation_date'])
fng['date'] = pd.to_datetime(fng['date'])
djia['date'] = pd.to_datetime(djia['observation_date'])
djca['date'] = pd.to_datetime(djca['observation_date'])

In [19]:
df = btc[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
df = df.merge(fed[['date', 'Actual']], on='date', how='left')
df = df.merge(sp500[['date', 'SP500']], on='date', how='left')
df = df.merge(fng[['date', 'fng_value']], on='date', how='left')
df = df.merge(djia[['date', 'DJIA']], on='date', how='left')
df = df.merge(djca[['date', 'DJCA']], on='date', how='left')

In [20]:
df['Actual'] = df['Actual'].astype(str).str.replace('%', '').astype(float).ffill()
df[['SP500', 'DJIA', 'DJCA', 'fng_value']] = df[['SP500', 'DJIA', 'DJCA', 'fng_value']].ffill()

In [21]:
df['price_change_pct'] = df['close'].pct_change() * 100
df['volatility'] = df['high'] - df['low']

In [28]:
df['sma_7'] = df['close'].rolling(window=7).mean()
df['sma_21'] = df['close'].rolling(window=21).mean()
df['rsi'] = calculate_rsi(df['close'], window=14)

In [29]:
df['sp500_change'] = df['SP500'].pct_change() * 100
df['djia_change'] = df['DJIA'].pct_change() * 100

In [30]:
df = df.dropna().reset_index(drop=True)

In [31]:
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

In [32]:
# features = [
#     'open', 'high', 'low', 'close', 'volume',
#     'price_change_pct', 'volatility', 'Actual',
#     'SP500', 'DJIA', 'DJCA', 'fng_value',
#     'sma_7', 'sma_21', 'rsi', 'sp500_change',
#     'djia_change', 'day_of_week', 'month'
# ]

# for feature in features:
#     for window in [1, 3, 5]:
#         df[f'{feature}_delta_{window}'] = df[feature].diff(window)


In [33]:
df.dropna()

,date,open,high,low,close,volume,Actual,SP500,fng_value,DJIA,DJCA,price_change_pct,volatility,sma_7,sma_21,rsi,sp500_change,djia_change,day_of_week,month
0,2020-02-18,9691.230469,10161.935547,9632.382812,10141.996094,47271023953,0.0175,3370.29,53.0,29232.19,9661.41,4.663022,529.552734,10072.649693,9765.820685,65.814413,-0.291998,-0.564289,1,2
1,2020-02-19,10143.798828,10191.675781,9611.223633,9633.386719,46992019710,0.0175,3386.15,50.0,29348.03,9680.12,-5.014884,580.452148,9973.697126,9780.904343,50.320471,0.470583,0.396275,2,2
2,2020-02-20,9629.325195,9643.216797,9507.900391,9608.475586,44925260237,0.0175,3373.23,44.0,29219.98,9679.08,-0.258592,135.316406,9887.139369,9785.641602,47.993384,-0.381554,-0.436315,3,2
3,2020-02-21,9611.782227,9723.014648,9589.743164,9686.441406,40930547513,0.0175,3337.75,44.0,28992.41,9602.49,0.811428,133.271484,9797.757254,9801.637416,48.196001,-1.051811,-0.778816,4,2
4,2020-02-22,9687.707031,9698.231445,9600.728516,9663.181641,35838025154,0.0175,3337.75,43.0,28992.41,9602.49,-0.240127,97.502930,9765.436802,9814.509161,46.622060,0.000000,0.000000,5,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1858,2025-03-21,84164.539062,84782.273438,83171.070312,84043.242188,19030452299,0.0550,5667.56,37.0,41985.35,13072.14,-0.147270,1611.203125,84111.664062,85075.511905,45.631484,0.082467,0.076347,4,3
1859,2025-03-22,84046.257812,84513.875000,83674.781250,83832.484375,9863214091,0.0550,5667.56,37.0,41985.35,13072.14,-0.250773,839.093750,84038.717634,84970.777158,46.195645,0.000000,0.000000,5,3
1860,2025-03-23,83831.898438,86094.781250,83794.914062,86054.375000,12594615537,0.0550,5667.56,37.0,41985.35,13072.14,2.650393,2299.867188,84535.101562,84580.587798,60.029150,0.000000,0.000000,6,3
1861,2025-03-24,86070.929688,88758.726562,85541.195312,87498.914062,34582604933,0.0550,5767.57,37.0,42583.32,13255.73,1.678635,3217.531250,85024.133929,84648.837426,66.878624,1.764604,1.424235,0,3


In [37]:
print(df.dropna().sample(5).to_string())

          date          open          high           low         close       volume  Actual    SP500  fng_value      DJIA      DJCA  price_change_pct   volatility         sma_7        sma_21        rsi  sp500_change  djia_change  day_of_week  month
286 2020-11-30  18178.322266  19749.263672  18178.322266  19625.835938  47728480399  0.0025  3621.63       88.0  29638.64   9901.74          7.967833  1570.941406  18231.335100  17486.178571  69.491536     -0.459549    -0.908481            0     11
111 2020-06-08   9760.063477   9782.306641   9675.885742   9771.489258  21486346312  0.0025  3232.39       53.0  27572.44   8962.47          0.129490   106.420898   9690.958984   9455.683222  63.386818      1.204159     1.702115            0      6
493 2021-06-25  34659.105469  35487.246094  31350.884766  31637.779297  40230904226  0.0025  4280.70       27.0  34433.84  11424.20         -8.726040  4136.361328  33645.680804  35848.213914  36.776937      0.333061     0.693105            4      6
621 

In [39]:
df.dropna().to_parquet("../data/04_feature/analytical_base_table_01.parquet", index=False)

In [2]:
df = pd.read_parquet("../data/04_feature/analytical_base_table_01.parquet")
df

,date,open,high,low,close,volume,Actual,SP500,fng_value,DJIA,DJCA,price_change_pct,volatility,sma_7,sma_21,rsi,sp500_change,djia_change,day_of_week,month
0,2020-02-18,9691.230469,10161.935547,9632.382812,10141.996094,47271023953,0.0175,3370.29,53.0,29232.19,9661.41,4.663022,529.552734,10072.649693,9765.820685,65.814413,-0.291998,-0.564289,1,2
1,2020-02-19,10143.798828,10191.675781,9611.223633,9633.386719,46992019710,0.0175,3386.15,50.0,29348.03,9680.12,-5.014884,580.452148,9973.697126,9780.904343,50.320471,0.470583,0.396275,2,2
2,2020-02-20,9629.325195,9643.216797,9507.900391,9608.475586,44925260237,0.0175,3373.23,44.0,29219.98,9679.08,-0.258592,135.316406,9887.139369,9785.641602,47.993384,-0.381554,-0.436315,3,2
3,2020-02-21,9611.782227,9723.014648,9589.743164,9686.441406,40930547513,0.0175,3337.75,44.0,28992.41,9602.49,0.811428,133.271484,9797.757254,9801.637416,48.196001,-1.051811,-0.778816,4,2
4,2020-02-22,9687.707031,9698.231445,9600.728516,9663.181641,35838025154,0.0175,3337.75,43.0,28992.41,9602.49,-0.240127,97.502930,9765.436802,9814.509161,46.622060,0.000000,0.000000,5,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1858,2025-03-21,84164.539062,84782.273438,83171.070312,84043.242188,19030452299,0.0550,5667.56,37.0,41985.35,13072.14,-0.147270,1611.203125,84111.664062,85075.511905,45.631484,0.082467,0.076347,4,3
1859,2025-03-22,84046.257812,84513.875000,83674.781250,83832.484375,9863214091,0.0550,5667.56,37.0,41985.35,13072.14,-0.250773,839.093750,84038.717634,84970.777158,46.195645,0.000000,0.000000,5,3
1860,2025-03-23,83831.898438,86094.781250,83794.914062,86054.375000,12594615537,0.0550,5667.56,37.0,41985.35,13072.14,2.650393,2299.867188,84535.101562,84580.587798,60.029150,0.000000,0.000000,6,3
1861,2025-03-24,86070.929688,88758.726562,85541.195312,87498.914062,34582604933,0.0550,5767.57,37.0,42583.32,13255.73,1.678635,3217.531250,85024.133929,84648.837426,66.878624,1.764604,1.424235,0,3


.